In [ ]:
# ============================================================
# TPU TRAINING - PyTorch on TPU
# ============================================================

# Install PyTorch TPU support
!pip install torch torchvision torch-xla

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time

# Initialize TPU
import torch_xla
import torch_xla.core.xla_model as xm

device = torch_xla.device()
print(f"Device: {device}")
print(f"Number of devices: {len(xm.get_xla_supported_devices())}")

# ... rest of the training script (same as GPU version)
# Just change device to xm.xla_device()

Device: xla:0
Number of devices: 1


In [3]:
# ============================================================
# COMPLETE TPU TRAINING - PyTorch on TPU
# Run this in a fresh notebook with TPU runtime
# ============================================================

# Step 1: Install PyTorch XLA (TPU support)
!pip install torch torchvision torch-xla

# Step 2: Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time
import os

# Step 3: Initialize TPU
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
import torch_xla.utils.utils as xu

# Get TPU device
device = torch_xla.device()
num_devices = len(xm.get_xla_supported_devices())

print("="*70)
print("🚀 ZenteiQ Assignment - TPU Training (PyTorch/XLA)")
print("="*70)
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ Device: {device}")
print(f"✅ Number of TPU cores: {num_devices}")
print("="*70)

# Step 4: Create synthetic dataset with PATTERNS
def create_pattern_data(batch_size=8, seq_len=64, vocab_size=100):
    """Create data with repeating patterns"""
    pattern = np.tile(np.arange(1, 11), (seq_len // 10 + 1))[:seq_len] % vocab_size
    noise = np.random.randint(0, 5, seq_len)
    pattern = (pattern + noise) % vocab_size
    data = np.tile(pattern[None, :], (batch_size, 1))
    return torch.tensor(data, dtype=torch.long)

# Step 5: Define model
class ImprovedModel(nn.Module):
    def __init__(self, vocab_size=100, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, vocab_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        embed = self.embedding(x)  # (batch, seq_len, embed_dim)
        h = embed[:, -1, :]  # (batch, embed_dim)
        h = self.relu(self.fc1(h))  # (batch, hidden_dim)
        logits = self.fc2(h)  # (batch, vocab_size)
        return logits

# Step 6: Initialize model on TPU
vocab_size = 100
embed_dim = 64
hidden_dim = 128
model = ImprovedModel(vocab_size, embed_dim, hidden_dim).to(device)

# Use XLA optimizer
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

print("\n🔥 Model initialized on TPU")
print("✅ Ready to train\n")

# Step 7: Training loop
print("🚀 STARTING TRAINING ON TPU")
print("="*70)
print("Step | Loss | Time (s)")
print("-"*70)

step_times = []
losses = []

for step in range(50):
    # Generate data on CPU then move to TPU
    x = create_pattern_data(batch_size=8, seq_len=64).to(device)
    y = create_pattern_data(batch_size=8, seq_len=64).to(device)

    start_time = time.perf_counter()

    # Forward pass
    logits = model(x)
    targets = y[:, -1]  # Predict next token
    loss = criterion(logits, targets)

    # Backward pass
    optimizer.zero_grad()
    loss.backward()

    # XLA: Mark step for TPU execution
    xm.optimizer_step(optimizer)

    step_time = time.perf_counter() - start_time

    step_times.append(step_time)
    losses.append(loss.item())

    if step % 5 == 0:
        print(f"{step:4d} | {loss.item():8.4f} | {step_time:.4f}s")

print("="*70)

# Step 8: Statistics
avg_time = np.mean(step_times)
total_time = np.sum(step_times)
loss_decrease = losses[0] - losses[-1]
pct_decrease = (loss_decrease / losses[0]) * 100 if losses[0] > 0 else 0

print(f"\n✅ TRAINING COMPLETE")
print("="*70)
print(f"📊 Device: {device}")
print(f"📊 TPU Cores: {num_devices}")
print(f"📊 PyTorch Version: {torch.__version__}")
print(f"📊 Average step time: {avg_time:.6f}s")
print(f"📊 Total training time: {total_time:.2f}s")
print(f"📊 Initial loss: {losses[0]:.4f}")
print(f"📊 Final loss: {losses[-1]:.4f}")
print(f"📊 Loss decrease: {loss_decrease:.4f} ({pct_decrease:.1f}%)")
print("="*70)

# Step 9: Save metrics
filename = "metrics_tpu_pytorch.txt"

with open(filename, "w") as f:
    f.write(f"Device: {device}\n")
    f.write(f"TPU Cores: {num_devices}\n")
    f.write(f"PyTorch Version: {torch.__version__}\n")
    f.write(f"Average step time: {avg_time:.6f}s\n")
    f.write(f"Total training time: {total_time:.2f}s\n")
    f.write(f"Initial loss: {losses[0]:.4f}\n")
    f.write(f"Final loss: {losses[-1]:.4f}\n")
    f.write(f"Loss decrease: {loss_decrease:.4f} ({pct_decrease:.1f}%)\n")
    f.write("\nStep,Time(s),Loss\n")
    for i, (t, l) in enumerate(zip(step_times, losses)):
        f.write(f"{i},{t:.6f},{l:.4f}\n")

print(f"\n📁 Metrics saved to: {filename}")

# Step 10: Display file content
print("\n" + "="*70)
print("📄 METRICS FILE CONTENT:")
print("="*70)
with open(filename, "r") as f:
    print(f.read())

# Step 11: Download file
try:
    from google.colab import files
    files.download(filename)
    print(f"\n📥 Downloaded: {filename}")
except:
    print("\n💡 To download: from google.colab import files; files.download(filename)")

🚀 ZenteiQ Assignment - TPU Training (PyTorch/XLA)
✅ PyTorch version: 2.9.0+cpu
✅ Device: xla:0
✅ Number of TPU cores: 1

🔥 Model initialized on TPU
✅ Ready to train

🚀 STARTING TRAINING ON TPU
Step | Loss | Time (s)
----------------------------------------------------------------------
   0 |   4.7204 | 0.0076s
   5 |   3.3650 | 0.0060s
  10 |   0.1181 | 0.0061s
  15 |   5.4074 | 0.0057s
  20 |   0.7332 | 0.0061s
  25 |   2.2122 | 0.0061s
  30 |   2.2761 | 0.0059s
  35 |   0.7996 | 0.0057s
  40 |   1.8942 | 0.0061s
  45 |   3.8460 | 0.0059s

✅ TRAINING COMPLETE
📊 Device: xla:0
📊 TPU Cores: 1
📊 PyTorch Version: 2.9.0+cpu
📊 Average step time: 0.006049s
📊 Total training time: 0.30s
📊 Initial loss: 4.7204
📊 Final loss: 1.5594
📊 Loss decrease: 3.1610 (67.0%)

📁 Metrics saved to: metrics_tpu_pytorch.txt

📄 METRICS FILE CONTENT:
Device: xla:0
TPU Cores: 1
PyTorch Version: 2.9.0+cpu
Average step time: 0.006049s
Total training time: 0.30s
Initial loss: 4.7204
Final loss: 1.5594
Loss decrease: 3

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📥 Downloaded: metrics_tpu_pytorch.txt
